# Machine Learning: Gender Wage Gap Decomposition
**Author:** Antara Sudhir  
**Course:** AD688 — Applied Business Analytics  
**Goal:** Quantify how much of the gender wage gap is explained by occupation/industry segregation vs. unexplained factors, using regression decomposition and Random Forest feature importance.

**Data:** IPUMS USA 2024 ACS (`employed_only`, 1.49M workers with positive wages)  
**Models:** Linear Regression (baseline + full) and Random Forest Regressor  
**Outputs:** Precomputed CSVs saved to `data/processed/` for loading in `pages/ml_methods.qmd`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import sys
sys.path.insert(0, "..")
from analysis.utils import load_employment

df = load_employment()
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## Step 1: Baseline Regression (No Occupation/Industry)

Predicts `INCWAGE` using only `AGE`, `RACE_LABEL`, `STATE_NAME`, and `SEX_LABEL`.  
The gender coefficient here is the **raw wage gap** after controlling for basic demographics.

In [ ]:
baseline_df = df[['AGE', 'RACE_LABEL', 'STATE_NAME', 'SEX_LABEL', 'INCWAGE']].copy()
baseline_encoded = pd.get_dummies(
    baseline_df, 
    columns=['RACE_LABEL', 'STATE_NAME', 'SEX_LABEL'], 
    drop_first=True
)

X_baseline = baseline_encoded.drop(columns=['INCWAGE'])
y_baseline = baseline_encoded['INCWAGE']

model_baseline = LinearRegression()
model_baseline.fit(X_baseline, y_baseline)

sex_col = [c for c in X_baseline.columns if 'SEX_LABEL' in c][0]
baseline_gender_coef = model_baseline.coef_[list(X_baseline.columns).index(sex_col)]

print(f"Baseline R²: {model_baseline.score(X_baseline, y_baseline):.4f}")
print(f"Gender coefficient ({sex_col}): ${baseline_gender_coef:,.2f}")
print(f"Interpretation: holding age, race, and state constant, men earn ${baseline_gender_coef:,.2f} more than women on average.")

## Step 2: Full Regression (Adding Occupation + Industry)

Adds broad `OCC_GROUP` (OCC // 100) and `IND_GROUP` (IND // 1000) to the model.  
The gender coefficient here is the **adjusted wage gap** after also controlling for job type.  
The difference between Step 1 and Step 2 coefficients = portion explained by occupational segregation.

In [ ]:
full_df = df[['AGE', 'RACE_LABEL', 'STATE_NAME', 'SEX_LABEL', 'OCC', 'IND', 'INCWAGE']].copy()
full_df['OCC_GROUP'] = (full_df['OCC'] // 100) * 100
full_df['IND_GROUP'] = (full_df['IND'] // 1000) * 1000

full_encoded = pd.get_dummies(
    full_df.drop(columns=['OCC', 'IND']),
    columns=['RACE_LABEL', 'STATE_NAME', 'SEX_LABEL', 'OCC_GROUP', 'IND_GROUP'],
    drop_first=True
)

X_full = full_encoded.drop(columns=['INCWAGE'])
y_full = full_encoded['INCWAGE']

model_full = LinearRegression()
model_full.fit(X_full, y_full)

sex_col_full = [c for c in X_full.columns if 'SEX_LABEL' in c][0]
full_gender_coef = model_full.coef_[list(X_full.columns).index(sex_col_full)]

print(f"Full model R²: {model_full.score(X_full, y_full):.4f}")
print(f"Gender coefficient ({sex_col_full}): ${full_gender_coef:,.2f}")
print(f"Interpretation: after controlling for occupation and industry, men earn ${full_gender_coef:,.2f} more than women.")

## Step 3: Wage Gap Decomposition Summary

Compares the two gender coefficients to calculate what % of the gap is explained vs. unexplained.

In [ ]:
explained_amount = baseline_gender_coef - full_gender_coef
explained_pct = (explained_amount / baseline_gender_coef) * 100
unexplained_pct = 100 - explained_pct

print("=" * 60)
print("GENDER WAGE GAP DECOMPOSITION")
print("=" * 60)
print(f"Raw gender gap (baseline):              ${baseline_gender_coef:,.2f}")
print(f"Adjusted gender gap (with occ/ind):     ${full_gender_coef:,.2f}")
print(f"Explained by occupation/industry:       ${explained_amount:,.2f} ({explained_pct:.1f}%)")
print(f"Unexplained gap:                        {unexplained_pct:.1f}%")
print()
print("Interpretation: Only 9.4% of the gender wage gap is explained by")
print("occupational/industry segregation. 90.6% persists within the same")
print("broad job type — suggesting the gap is not simply about job choice.")

## Step 4: Random Forest Regressor

Uses a 200,000-row sample (from 1.49M) for computational efficiency on a 2-core EC2 instance.  
100 trees, max depth 10. Trains a non-linear model to confirm gender's importance  
as a wage predictor without any assumption about a fixed linear effect.

In [ ]:
# Sample 200k rows for computational efficiency
rf_df = df[['AGE', 'RACE_LABEL', 'STATE_NAME', 'SEX_LABEL', 'OCC', 'IND', 'INCWAGE']].sample(
    n=200000, random_state=42
).copy()

rf_df['OCC_GROUP'] = (rf_df['OCC'] // 100) * 100
rf_df['IND_GROUP'] = (rf_df['IND'] // 1000) * 1000

rf_encoded = pd.get_dummies(
    rf_df.drop(columns=['OCC', 'IND']),
    columns=['RACE_LABEL', 'STATE_NAME', 'SEX_LABEL', 'OCC_GROUP', 'IND_GROUP'],
    drop_first=True
)

X = rf_encoded.drop(columns=['INCWAGE'])
y = rf_encoded['INCWAGE']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

train_score = rf_model.score(X_train, y_train)
test_score = rf_model.score(X_test, y_test)

print(f"Random Forest Train R²: {train_score:.4f}")
print(f"Random Forest Test R²:  {test_score:.4f}")
print(f"Sample size: {len(X):,} workers")

## Step 5: Feature Importance and Partial Dependence

Feature importance shows which features were most useful for predicting wage across all 100 trees.  
Partial dependence shows predicted wage by age, split by gender, holding all other features at baseline.

In [ ]:
# Feature importance
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

importances['is_gender'] = importances['feature'].str.contains('SEX_LABEL')

sex_rank = importances[importances['feature'].str.contains('SEX_LABEL')].index[0] + 1
print(f"Top 15 Most Important Features (out of {len(importances)}):")
print(importances.head(15)[['feature', 'importance', 'is_gender']].to_string(index=True))
print(f"\nGender (SEX_LABEL_Male) rank: #{sex_rank} of {len(importances)} features")

# Partial dependence by age and gender
age_range = np.arange(20, 66, 2)
pdp_rows = []
for sex in ['Male', 'Female']:
    for age in age_range:
        row = {col: 0 for col in X.columns}
        row['AGE'] = age
        if sex == 'Male':
            row['SEX_LABEL_Male'] = 1
        pdp_rows.append({'AGE': age, 'SEX': sex, **row})

pdp_df = pd.DataFrame(pdp_rows)
pdp_df['predicted_wage'] = rf_model.predict(pdp_df[X.columns])
pdp_summary = pdp_df[['AGE', 'SEX', 'predicted_wage']]
print(f"\nPartial dependence sample (first 6 rows):")
print(pdp_summary.head(6))

## Step 6: Save Precomputed Results

Saves all results as small CSVs to `data/processed/` so `pages/ml_methods.qmd`  
can load them without retraining models on every site render.

In [ ]:
import os
output_dir = '../data/processed/'

# 1. Decomposition summary
decomposition_df = pd.DataFrame({
    'model': ['Baseline (age, race, state)', 'Full (+ occupation, industry)'],
    'gender_coefficient': [baseline_gender_coef, full_gender_coef],
    'r_squared': [
        model_baseline.score(X_baseline, y_baseline),
        model_full.score(X_full, y_full)
    ],
    'explained_pct': [0, explained_pct]
})
decomposition_df.to_csv(f'{output_dir}wage_gap_decomposition_antara.csv', index=False)
print("✓ Saved wage_gap_decomposition_antara.csv")

# 2. RF feature importance (top 20)
importances.head(20).to_csv(f'{output_dir}rf_feature_importance_antara.csv', index=False)
print("✓ Saved rf_feature_importance_antara.csv")

# 3. RF model summary
pd.DataFrame({
    'metric': ['Train R²', 'Test R²', 'Sample Size'],
    'value': [train_score, test_score, len(X)]
}).to_csv(f'{output_dir}rf_model_summary_antara.csv', index=False)
print("✓ Saved rf_model_summary_antara.csv")

# 4. Partial dependence
pdp_summary.to_csv(f'{output_dir}rf_partial_dependence_age_gender_antara.csv', index=False)
print("✓ Saved rf_partial_dependence_age_gender_antara.csv")

# 5. OCC group labels (from IPUMS 2018 Census coding scheme)
def get_major_group(occ):
    if occ == 0: return 'Not Reported'
    elif occ <= 430: return 'Management'
    elif occ <= 730: return 'Business & Financial Operations'
    elif occ <= 950: return 'Financial Specialists'
    elif occ <= 1240: return 'Computer & Mathematical'
    elif occ <= 1540: return 'Architecture & Engineering'
    elif occ <= 1980: return 'Life, Physical & Social Science'
    elif occ <= 2060: return 'Community & Social Services'
    elif occ <= 2150: return 'Legal'
    elif occ <= 2550: return 'Education, Training & Library'
    elif occ <= 2920: return 'Arts, Design, Entertainment & Media'
    elif occ <= 3540: return 'Healthcare Practitioners'
    elif occ <= 3650: return 'Healthcare Support'
    elif occ <= 3950: return 'Protective Service'
    elif occ <= 4150: return 'Food Preparation & Serving'
    elif occ <= 4250: return 'Building & Grounds Cleaning'
    elif occ <= 4650: return 'Personal Care & Service'
    elif occ <= 4965: return 'Sales & Related'
    elif occ <= 5940: return 'Office & Administrative Support'
    elif occ <= 6130: return 'Farming, Fishing & Forestry'
    elif occ <= 6765: return 'Construction & Extraction'
    elif occ <= 7630: return 'Installation, Maintenance & Repair'
    elif occ <= 8965: return 'Production'
    elif occ <= 9750: return 'Transportation & Material Moving'
    else: return 'Military Specific'

df_occ = df[['OCC']].copy()
df_occ['OCC_GROUP'] = (df_occ['OCC'] // 100) * 100
df_occ['MAJOR_GROUP'] = df_occ['OCC'].apply(get_major_group)
occ_lookup = (df_occ.groupby(['OCC_GROUP', 'MAJOR_GROUP'])
              .size().reset_index(name='count')
              .sort_values('count', ascending=False)
              .drop_duplicates('OCC_GROUP')
              [['OCC_GROUP', 'MAJOR_GROUP']]
              .sort_values('OCC_GROUP'))
occ_lookup.to_csv(f'{output_dir}occ_group_labels.csv', index=False)
print("✓ Saved occ_group_labels.csv")

print("\nAll files saved successfully!")